# Monte Carlo approaches to prediction and control

In this notebook, you will implement the Monte Carlo approaches to prediction and control described in [Sutton and Barto's book, Introduction to Reinforcement Learning](http://incompleteideas.net/book/the-book-2nd.html). We will use the grid ```World``` class from the previous lecture, but now without relying on knowledge of the task dynamics, that is, without relying on knowledge about transition probabilities.

### Install dependencies

In [513]:
! pip install numpy pandas

'pip' is not recognized as an internal or external command,
operable program or batch file.


### Imports

In [514]:
import numpy as np
import random
import sys          # We use sys to get the max value of a float
import pandas as pd # We only use pandas for displaying tables nicely
from typing import Callable
pd.options.display.float_format = '{:,.3f}'.format

### ```World``` class and globals

The ```World``` is a grid represented as a two-dimensional array of characters where each character can represent free space, an obstacle, or a terminal. Each non-obstacle cell is associated with a reward that an agent gets for moving to that cell (can be 0). The size of the world is _width_ $\times$ _height_ characters.

A _state_ is a tuple $(x,y)$.

An empty world is created in the ```__init__``` method. Obstacles, rewards and terminals can then be added with ```add_obstacle``` and ```add_reward```.

To calculate the next state of an agent (that is, an agent is in some state $s = (x,y)$ and performs and action, $a$), ```get_next_state()```should be called.

__Note that ```get_state_transition_probabilities``` has been removed and an agent must now rely on experience interacting with a world to learn.__

In [515]:
# Globals:
ACTIONS = ("up", "down", "left", "right")

# Rewards, terminals and obstacles are characters:
REWARDS = {" ": 0, ".": 0.1, "+": 10, "-": -10}
TERMINALS = ("+", "-") # Note a terminal should also have a reward assigned
OBSTACLES = ("#")

# Discount factor
gamma = 1

# The probability of a random move:
rand_move_probability = 0

class World:
  def __init__(self, width, height):
    self.width = width
    self.height = height
    # Create an empty world where the agent can move to all cells
    self.grid = np.full((width, height), ' ', dtype='U1')

  def add_obstacle(self, start_x, start_y, end_x=None, end_y=None):
    """
    Create an obstacle in either a single cell or rectangle.
    """
    if end_x == None: end_x = start_x
    if end_y == None: end_y = start_y

    self.grid[start_x:end_x + 1, start_y:end_y + 1] = OBSTACLES[0]

  def add_reward(self, x, y, reward):
    assert reward in REWARDS, f"{reward} not in {REWARDS}"
    self.grid[x, y] = reward

  def add_terminal(self, x, y, terminal):
    assert terminal in TERMINALS, f"{terminal} not in {TERMINALS}"
    self.grid[x, y] = terminal

  def is_obstacle(self, x, y):
    if x < 0 or x >= self.width or y < 0 or y >= self.height:
      return True
    else:
      return self.grid[x ,y] in OBSTACLES

  def is_terminal(self, x, y):
    return self.grid[x ,y] in TERMINALS

  def get_reward(self, x, y):
    """
    Return the reward associated with a given location
    """
    return REWARDS[self.grid[x, y]]

  def get_next_state(self, current_state, action):
    """
    Get the next state given a current state and an action. The outcome can be
    stochastic  where rand_move_probability determines the probability of
    ignoring the action and performing a random move.
    """
    assert action in ACTIONS, f"Unknown acion {action} must be one of {ACTIONS}"

    x, y = current_state

    # If our current state is a terminal, there is no next state
    if self.grid[x, y] in TERMINALS:
      return None

    # Check of a random action should be performed:
    if np.random.rand() < rand_move_probability:
      action = np.random.choice(ACTIONS)

    if action == "up":      y -= 1
    elif action == "down":  y += 1
    elif action == "left":  x -= 1
    elif action == "right": x += 1

    # If the next state is an obstacle, stay in the current state
    return (x, y) if not self.is_obstacle(x, y) else current_state


## Basic example: Generating episodes

An episode is the series of states, actions and rewards reflecting an agent's experience interacting with the environment. An episode starts with an agent being placed at some initial state and continues till the agent reaches a terminal state.  To generate episodes, we first need a world and a policy:


In [516]:
world = World(2, 3)

# Since we only focus on episodic tasks, we must have a terminal state that the
# agent eventually reaches
world.add_terminal(1, 2, "+")

def equiprobable_random_policy(x, y):
  return { k:1/len(ACTIONS) for k in ACTIONS }

print(world.grid.T)

[[' ' ' ']
 [' ' ' ']
 [' ' '+']]


To generate an episode, we need to provide a ```World```, a policy, and a start state.

In each step, we do the following:
1. perform one of the actions (weighted random) returned by the policy for the giving state
2. get the reward and add a new entry to the episode $[S_t, A_t, R_{t+1}]$
3. move the agent to the next state

When a terminal state is reached, we return all the $[[S_0, A_0, R_1], ..., [S_{T}, A_T, R_{T+1}]]$ observed in the episode.

In [517]:
# def generate_episode(world, policy, start_state, random_first_move=0):
#     current_state = start_state
#     episode = []
#     first_move = True
#     while not world.is_terminal(*current_state):
        
#         possible_actions = policy(*current_state)
#         if first_move and random_first_move:
#             action = [ACTIONS[random.randint(0, len(ACTIONS))], "fuck you"]
#             first_move = False
#         else:
#             action = random.choices(population=list(possible_actions.keys()),
#                                     weights=possible_actions.values(), k=1)
#         next_state = world.get_next_state(current_state, action[0])
#         reward = world.get_reward(*next_state)
#         episode.append([current_state, action[0], reward])
#         current_state = next_state

#     return episode

def generate_episode(world, policy, start_state, random_first_move=0, max_steps=1000000):
    current_state = start_state
    episode = []
    first_move = True
    steps = 0
    
    while not world.is_terminal(*current_state) and steps < max_steps:
        possible_actions = policy(*current_state)
        if first_move and random_first_move:
            action = ACTIONS[random.randint(0, len(ACTIONS) - 1)]
            first_move = False
        else:
            action = random.choices(population=list(possible_actions.keys()),
                                    weights=possible_actions.values(), k=1)[0]
        
        next_state = world.get_next_state(current_state, action)
        reward = world.get_reward(*next_state)
        episode.append([current_state, action, reward])
        current_state = next_state
        steps += 1

    return episode

In [518]:
def visualize_best_actions(world, V):
    """Show the best action (highest value) for each state"""
    best_actions = np.full((world.width, world.height), ' ', dtype='U10')
    
    # Symbols for directions
    action_symbols = {
        'up': '↑',
        'down': '↓', 
        'left': '←',
        'right': '→'
    }
    
    for y in range(world.height):
        for x in range(world.width):
            if world.is_obstacle(x, y):
                best_actions[x, y] = '#'
            elif world.is_terminal(x, y):
                best_actions[x, y] = world.grid[x, y]
            else:
                # Find neighboring state with highest value
                neighbors = []
                for action in ACTIONS:
                    next_state = world.get_next_state((x, y), action)
                    if next_state:
                        neighbors.append((action, V[next_state[0], next_state[1]]))
                
                if neighbors:
                    best_action = max(neighbors, key=lambda item: item[1])[0]
                    best_actions[x, y] = action_symbols[best_action]
    
    print("Best action at each state (arrows show direction):")
    display(pd.DataFrame(best_actions.T))

In [519]:
def visualize_policy_grid(policy_grid, visited=None):
    """Convert policy grid from action names to arrow symbols"""
    action_symbols = {
        'up': '↑',
        'down': '↓', 
        'left': '←',
        'right': '→',
        '+': '+',
        '-': '-',
        '#': '#'
    }
    
    arrow_grid = np.full(policy_grid.shape, ' ', dtype='U10')
    
    for y in range(policy_grid.shape[1]):
        for x in range(policy_grid.shape[0]): 
            if (visited is not None) and (visited[x, y] == 0):
                continue
            action = policy_grid[x, y]
            arrow_grid[x, y] = action_symbols.get(action, action)
    
    print("Policy visualization (arrows show best action):")
    display(pd.DataFrame(arrow_grid.T))

Now, we can try to generate a couple of episodes and print the result:

In [520]:
for i in range(5):
    print(f"Episode {i}:")
    episode = generate_episode(world, equiprobable_random_policy, (0, 0))
    print(pd.DataFrame(episode, columns=["State", "Action", "Reward"]), end="\n\n")

Episode 0:
     State Action  Reward
0   (0, 0)   left       0
1   (0, 0)  right       0
2   (1, 0)     up       0
3   (1, 0)     up       0
4   (1, 0)  right       0
5   (1, 0)     up       0
6   (1, 0)   left       0
7   (0, 0)   left       0
8   (0, 0)   left       0
9   (0, 0)  right       0
10  (1, 0)   down       0
11  (1, 1)  right       0
12  (1, 1)  right       0
13  (1, 1)   left       0
14  (0, 1)   left       0
15  (0, 1)  right       0
16  (1, 1)   down      10

Episode 1:
     State Action  Reward
0   (0, 0)  right       0
1   (1, 0)   down       0
2   (1, 1)     up       0
3   (1, 0)  right       0
4   (1, 0)     up       0
..     ...    ...     ...
62  (1, 0)   left       0
63  (0, 0)  right       0
64  (1, 0)   down       0
65  (1, 1)  right       0
66  (1, 1)   down      10

[67 rows x 3 columns]

Episode 2:
    State Action  Reward
0  (0, 0)  right       0
1  (1, 0)   left       0
2  (0, 0)  right       0
3  (1, 0)   down       0
4  (1, 1)  right       0
5  (1, 1)   

In [521]:
def equiprobable_random_policy(x, y):
    return { k : 1 / len(ACTIONS) for k in ACTIONS }
print(equiprobable_random_policy(0, 0))

{'up': 0.25, 'down': 0.25, 'left': 0.25, 'right': 0.25}


### Exercise: Implement Monte Carlo-based prediction for state values

You should implement first-visit MC prediction for estimating $V≈v_\pi$. See page 92 of [Introduction to Reinforcement Learning](http://incompleteideas.net/book/the-book-2nd.html).


In [522]:
def first_visit_mc(world, policy, no_of_episodes = 10):
    v = np.full((world.width, world.height), 0.0)
    Returns = []
    for x in range(world.width):
        Returns.append([])
        for _ in range(world.height):
            Returns[x].append([]) # can be called with Returns[x][y]

    for _ in range(no_of_episodes):
        episode = generate_episode(world=world, policy=policy, start_state=(0, 0))
        g = 0.0
        for idx, (s, _, r) in reversed(list(enumerate(episode))):
            x, y = s
            g = r + gamma * g
            if s not in [state for state, _, _ in episode[:idx]]:
                Returns[x][y].append(g)
                v[s] = np.mean(Returns[x][y])
    return v

first_visit_world = World(4, 4)

first_visit_world.add_terminal(3, 3, "+")

display(pd.DataFrame(world.grid.T))

rand_move_probability = 0
gamma = 0.9
v = first_visit_mc(world, equiprobable_random_policy, 10000)

display(pd.DataFrame(v.T))

visualize_best_actions(world, v)

,0,1
0,,
1,,
2,,+


,0,1
0,3.258,3.646
1,4.380,5.500
2,6.360,0.000


Best action at each state (arrows show direction):


,0,1
0,↓,↓
1,↓,→
2,↓,+


First, try your algorithm on the small $2\times3$ world above using an equiprobable policy and $\gamma = 0.9$. Depending on the number of episodes you use, you should get close to the true values:

<table class="dataframe" border="1">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>0</th>
      <th>1</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>3.283</td>
      <td>3.616</td>
    </tr>
    <tr>
      <th>1</th>
      <td>4.409</td>
      <td>5.556</td>
    </tr>
    <tr>
      <th>2</th>
      <td>6.349</td>
      <td>0.000</td>
    </tr>
  </tbody>
</table>




In [523]:
gamma = 0.9

### TODO: Implement your code here

Try to run your MC prediction code on worlds of different sizes (be careful not to make your world too large or you should have multiple terminals that an agent is likely to hit, otherwise it may take too long). You can try to change the policy as well, but rememeber that the agent **must** eventually reach a terminal state under any policy that you try.

In [524]:
### TODO: Implement your code here

### Exercise: Implement Monte Carlo-based prediction for state-action values

There is one more step that has to be in place before we can start to optimize a policy: estimating state-action values, $q_\pi(s,a)$, based on experience. Above where we estimated $v_\pi$, we only needed to keep track of the average return observed for _each state_. However, in order to estimate state-action values, we need to compute the average return observed for _each state-action_ pair.

That is, for every state $(0,0), (0,1), (0,2)...$ we need to compute different estimates for the four actions ```[ "up", "down", "left", "right" ]```

In [525]:
def exploring_starts_mc(world, no_of_episodes = 10):
    policy = equiprobable_random_policy # policy(x,y)
    policy_grid = np.full((world.width, world.height), "", dtype=object) # need object type to have string
    q = np.full((world.width, world.height, len(ACTIONS)), 0.0) # q[x,y,a_idx]
    Returns = [] # Returns[x][y][a]
    for x in range(world.width):
        Returns.append([]) # state x
        for y in range(world.height):
            Returns[x].append([]) # state y
            for _ in range(len(ACTIONS)):
                Returns[x][y].append([]) # actions

    for _ in range(no_of_episodes):
        x, y = random.randint(0, world.width), random.randint(0, world.height) # random start
        episode = generate_episode(world=world, policy=policy, start_state=(x, y), random_first_move=1)
        g = 0.0
        for idx, (s, a, r) in reversed(list(enumerate(episode))):
            x, y = s
            g = r + gamma * g
            if (s, a) not in [(state, action) for state, action, _ in episode[:idx]]:
                Returns[x][y][a_idx].append(g)
                a_idx = ACTIONS.index(a)
                q[x, y, a_idx] = np.mean(Returns[x][y][a_idx])

                policy_grid[x, y] = ACTIONS[np.argmax(q[x, y, :])]
                
        policy = lambda x, y: {k : 1 if k == policy_grid[x,y] else 0 for k in ACTIONS}
    return q

display(pd.DataFrame(world.grid.T))

rand_move_probability = 0
gamma = 0.9
v = first_visit_mc(world, equiprobable_random_policy, 1000)

display(pd.DataFrame(v.T))

# visualize_best_actions(world, v)

,0,1
0,,
1,,
2,,+


,0,1
0,3.277,3.658
1,4.409,5.635
2,6.339,0.000


Try to experiment with your implementation by running it on different world sizes (be careful not to make your world too large or you should have multiple terminals that an agent is likely to hit, otherwise it may take too long), and try to experiment with different numbers of episodes:

In [526]:
### TODO: Implement your code here

### Exercise: Implement on-policy Monte Carlo-based control with an $\epsilon$-soft policy

You are now ready to implement MC-based control (see page 101 of [Introduction to Reinforcement Learning](http://incompleteideas.net/book/the-book-2nd.html) for the algorithm).

In your implementation, you need to update the state-action estimates like in the exercise above, but now, you also need implement an $ϵ$-soft policy that you can modify. How could you do that?

_Hint_: You can either represent your policy explicitly. That is, for each state $(x,y)$ you have a ```dict``` with actions and their probabilities which you then update each time you step through an episode. When the policy is called, it then just returns the ```dict``` with action probablities corresponding to the current state.

Alternatively, you can compute the action probabilities when your policy is called based on the current action-values estimates.

In [527]:
gamma = 0.9
epsilon = 0.1

def soft_first_visit_mc(world, epsilon=0.1, no_of_episodes = 10, start_state=(0, 0)) -> tuple[np.ndarray, Callable]: # on-policy
    q = np.full((world.width, world.height, len(ACTIONS)), 0.0) # q[x,y,a_idx]
    Returns = [] # Returns[x][y][a]

    policy_grid = np.full((world.width, world.height), "down", dtype=object) # need object type to have string
    visited_mirror = np.full((world.width, world.height), 0)
    e = epsilon
    policy = lambda x, y, q_func=q: {k : 1 - e + e/len(ACTIONS) if k == ACTIONS[np.argmax(q_func[x, y, :])] else e/len(ACTIONS) for k in ACTIONS}

    for x in range(world.width):
        Returns.append([]) # state x
        for y in range(world.height):
            Returns[x].append([]) # state y
            for _ in range(len(ACTIONS)):
                Returns[x][y].append([]) # actions
                if world.is_terminal(x, y):
                    policy_grid[x, y] = "+"
                    visited_mirror[x, y] = 8
                if world.is_obstacle(x, y):
                    policy_grid[x, y] = "#"
                    visited_mirror[x, y] = 4

    for no in range(no_of_episodes):
        episode = generate_episode(world=world, policy=policy, start_state=start_state)
        g = 0.0
        for idx, (s, a, r) in reversed(list(enumerate(episode))):
            x, y = s
            g = r + gamma * g
            if (s, a) not in [(state, action) for state, action, _ in episode[:idx]]:
                a_idx = ACTIONS.index(a)
                Returns[x][y][a_idx].append(g)
                q[x, y, a_idx] = np.mean(Returns[x][y][a_idx])
                policy_grid[x, y] = ACTIONS[np.argmax(q[x, y, :])] # A* (a_greedy)
                visited_mirror[x, y] = 1

    return q, policy, policy_grid, visited_mirror

# big map test
REWARDS = {" ": 0, "#": -1, "+": 10, "-": -10}

big_map = World(10, 10)
big_map.add_terminal(9, 0, "+")
big_map.add_terminal(9, 9, "+")
big_map.add_terminal(0, 9, "+")
big_map.add_terminal(0, 0, "+")
# big_map.add_terminal(9, 1, "-")
# big_map.add_terminal(9, 8, "-")
obstacles = [(1, 1), (2, 1), (3, 1), (7, 0), (2, 2), (2, 3), (7, 2), (7, 3), (2, 8), (2, 9), (7, 7), (7, 8)]
for x, y in obstacles:
    big_map.add_obstacle(x, y)

display(pd.DataFrame(big_map.grid.T))

rand_move_probability = 0
gamma = 0.9
q, policy, policy_grid, visited_mirror = soft_first_visit_mc(big_map, epsilon=(2/3), no_of_episodes=1000, start_state=(5,5))
print(policy_grid)
display(pd.DataFrame(policy_grid.T))

visualize_policy_grid(policy_grid)
visualize_policy_grid(policy_grid, visited_mirror)
display(pd.DataFrame(visited_mirror.T))

,0,1,2,3,4,5,6,7,8,9
0,+,,,,,,,#,,+
1,,#,#,#,,,,,,
2,,,#,,,,,#,,
3,,,#,,,,,#,,
4,,,,,,,,,,
5,,,,,,,,,,
6,,,,,,,,,,
7,,,,,,,,#,,
8,,,#,,,,,#,,
9,+,,#,,,,,,,+


[['+' 'up' 'up' 'up' 'up' 'down' 'down' 'down' 'down' '+']
 ['left' '#' 'left' 'up' 'up' 'down' 'down' 'down' 'down' 'left']
 ['left' '#' '#' '#' 'left' 'left' 'left' 'left' '#' '#']
 ['right' '#' 'right' 'down' 'left' 'left' 'left' 'left' 'up' 'left']
 ['right' 'down' 'right' 'up' 'right' 'left' 'left' 'left' 'left' 'up']
 ['down' 'right' 'right' 'right' 'right' 'right' 'right' 'left' 'left'
  'left']
 ['down' 'right' 'up' 'up' 'right' 'right' 'right' 'up' 'down' 'right']
 ['#' 'right' '#' '#' 'right' 'down' 'right' '#' '#' 'right']
 ['right' 'up' 'up' 'up' 'up' 'right' 'down' 'down' 'right' 'right']
 ['+' 'up' 'up' 'up' 'up' 'down' 'down' 'down' 'down' '+']]


,0,1,2,3,4,5,6,7,8,9
0,+,left,left,right,right,down,down,#,right,+
1,up,#,#,#,down,right,right,right,up,up
2,up,left,#,right,right,right,up,#,up,up
3,up,up,#,down,up,right,up,#,up,up
4,up,up,left,left,right,right,right,right,up,up
5,down,down,left,left,left,right,right,down,right,down
6,down,down,left,left,left,right,right,right,down,down
7,down,down,left,left,left,left,up,#,down,down
8,down,down,#,up,left,left,down,#,right,down
9,+,left,#,left,up,left,right,right,right,+


Policy visualization (arrows show best action):


,0,1,2,3,4,5,6,7,8,9
0,+,←,←,→,→,↓,↓,#,→,+
1,↑,#,#,#,↓,→,→,→,↑,↑
2,↑,←,#,→,→,→,↑,#,↑,↑
3,↑,↑,#,↓,↑,→,↑,#,↑,↑
4,↑,↑,←,←,→,→,→,→,↑,↑
5,↓,↓,←,←,←,→,→,↓,→,↓
6,↓,↓,←,←,←,→,→,→,↓,↓
7,↓,↓,←,←,←,←,↑,#,↓,↓
8,↓,↓,#,↑,←,←,↓,#,→,↓
9,+,←,#,←,↑,←,→,→,→,+


Policy visualization (arrows show best action):


,0,1,2,3,4,5,6,7,8,9
0,+,←,←,→,→,↓,↓,#,→,+
1,↑,#,#,#,↓,→,→,→,↑,↑
2,↑,←,#,→,→,→,↑,#,↑,↑
3,↑,↑,#,↓,↑,→,↑,#,↑,↑
4,↑,↑,←,←,→,→,→,→,↑,↑
5,↓,↓,←,←,←,→,→,↓,→,↓
6,↓,↓,←,←,←,→,→,→,↓,↓
7,↓,↓,←,←,←,←,↑,#,↓,↓
8,↓,↓,#,↑,←,←,↓,#,→,↓
9,+,←,#,←,↑,←,→,→,→,+


,0,1,2,3,4,5,6,7,8,9
0,8,1,1,1,1,1,1,4,1,8
1,1,4,4,4,1,1,1,1,1,1
2,1,1,4,1,1,1,1,4,1,1
3,1,1,4,1,1,1,1,4,1,1
4,1,1,1,1,1,1,1,1,1,1
5,1,1,1,1,1,1,1,1,1,1
6,1,1,1,1,1,1,1,1,1,1
7,1,1,1,1,1,1,1,4,1,1
8,1,1,4,1,1,1,1,4,1,1
9,8,1,4,1,1,1,1,1,1,8


In [528]:
# #visual glamour
# def soft_first_visit_mc(world, epsilon=0.1, no_of_episodes = 10, start_state=(0, 0)):
#     q = np.full((world.width, world.height, len(ACTIONS)), 0.0) # q[x,y,a_idx]
#     Returns = [] # Returns[x][y][a]
#     e = epsilon
#     policy = lambda x, y, q_func=q: {k : 1 - e + e/len(ACTIONS) 
#                                      if k == ACTIONS[np.argmax(q_func[x, y, :])] 
#                                      else e/len(ACTIONS) for k in ACTIONS}
#     for x in range(world.width):
#         Returns.append([])
#         for y in range(world.height):
#             Returns[x].append([])
#             for _ in range(len(ACTIONS)):
#                 Returns[x][y].append([]) # for actions

#     for _ in range(no_of_episodes):
#         episode = generate_episode(world=world, policy=policy, start_state=start_state)
#         g = 0.0
#         for idx, (s, a, r) in reversed(list(enumerate(episode))):
#             x, y = s
#             g = r + gamma * g
#             if (s, a) not in [(state, action) for state, action, _ in episode[:idx]]:
#                 a_idx = ACTIONS.index(a)
#                 Returns[x][y][a_idx].append(g)
#                 q[x, y, a_idx] = np.mean(Returns[x][y][a_idx])

#     return q, policy

# def first_visit_mc(world, policy, no_of_episodes = 1000, start_state=(0, 0)):
#     v = np.full((world.width, world.height), 0.0) # V[x, y]
#     Returns = []
#     for x in range(world.width):
#         Returns.append([])
#         for _ in range(world.height):
#             Returns[x].append([]) # Returns[x][y]

#     for _ in range(no_of_episodes):
#         episode = generate_episode(world=world, policy=policy, start_state=start_state)
#         g = 0.0
#         for idx, (s, _, r) in reversed(list(enumerate(episode))):
#             x, y = s
#             g = r + gamma * g
#             if s not in [state for state, _, _ in episode[:idx]]:
#                 Returns[x][y].append(g)
#                 v[s] = np.mean(Returns[x][y])
#     return v

# def exploring_starts_mc(world, no_of_episodes = 1000):
#     policy = equiprobable_random_policy # policy(x,y)
#     q = np.full((world.width, world.height, len(ACTIONS)), 0.0) # q[x,y,a_idx]
#     policy = lambda x, y, q_func=q: {k : 1 if k == ACTIONS[np.argmax(q_func[x, y, :])] else 0 for k in ACTIONS}
    
#     Returns = [] # Returns[x][y][a]
#     for x in range(world.width):
#         Returns.append([])
#         for y in range(world.height):
#             Returns[x].append([])
#             for _ in range(len(ACTIONS)):
#                 Returns[x][y].append([]) # for actions

#     for _ in range(no_of_episodes):
#         x, y = random.randint(0, world.width), random.randint(0, world.height) # random start
#         episode = generate_episode(world=world, policy=policy, start_state=(x, y), random_first_move=1)
#         g = 0.0
#         for idx, (s, a, r) in reversed(list(enumerate(episode))):
#             x, y = s
#             g = r + gamma * g
#             if (s, a) not in [(state, action) for state, action, _ in episode[:idx]]:
#                 Returns[x][y][a_idx].append(g)
#                 a_idx = ACTIONS.index(a)
#                 q[x, y, a_idx] = np.mean(Returns[x][y][a_idx])
#     return q, policy

In [529]:
import matplotlib.pyplot as plt
import time
from collections import defaultdict

# Benchmark and comparison code for Monte Carlo algorithms

# Test world setup
def create_test_world():
    """Create a standard test world for benchmarking"""
    test_world = World(10, 10)
    test_world.add_terminal(9, 0, "+")
    test_world.add_terminal(9, 9, "+")
    test_world.add_terminal(0, 9, "+")
    test_world.add_terminal(0, 0, "+")
    obstacles = [(1, 1), (2, 1), (3, 1), (7, 0), (2, 2), (2, 3), (7, 2), (7, 3), (2, 8), (2, 9), (7, 7), (7, 8)]
    for x, y in obstacles:
        test_world.add_obstacle(x, y)
    return test_world

# Modified algorithms to track performance metrics
def soft_first_visit_mc_tracked(world, epsilon=0.1, gamma_val=0.9, no_of_episodes=10, start_state=(0, 0)):
    """Modified soft_first_visit_mc that tracks episode lengths and runtime"""
    global gamma
    gamma = gamma_val
    
    q = np.full((world.width, world.height, len(ACTIONS)), 0.0)
    Returns = []
    policy_grid = np.full((world.width, world.height), "down", dtype=object)
    visited_mirror = np.full((world.width, world.height), 0)
    e = epsilon
    policy = lambda x, y, q_func=q: {k : 1 - e + e/len(ACTIONS) if k == ACTIONS[np.argmax(q_func[x, y, :])] else e/len(ACTIONS) for k in ACTIONS}

    for x in range(world.width):
        Returns.append([])
        for y in range(world.height):
            Returns[x].append([])
            for _ in range(len(ACTIONS)):
                Returns[x][y].append([])
                if world.is_terminal(x, y):
                    policy_grid[x, y] = "+"
                    visited_mirror[x, y] = 8
                if world.is_obstacle(x, y):
                    policy_grid[x, y] = "#"
                    visited_mirror[x, y] = 4

    episode_lengths = []
    cumulative_rewards = []
    
    start_time = time.time()
    
    for no in range(no_of_episodes):
        episode = generate_episode(world=world, policy=policy, start_state=start_state)
        episode_lengths.append(len(episode))
        
        total_reward = sum([r for _, _, r in episode])
        cumulative_rewards.append(total_reward)
        
        g = 0.0
        for idx, (s, a, r) in reversed(list(enumerate(episode))):
            x, y = s
            g = r + gamma * g
            if (s, a) not in [(state, action) for state, action, _ in episode[:idx]]:
                a_idx = ACTIONS.index(a)
                Returns[x][y][a_idx].append(g)
                q[x, y, a_idx] = np.mean(Returns[x][y][a_idx])
                policy_grid[x, y] = ACTIONS[np.argmax(q[x, y, :])]
                visited_mirror[x, y] = 1
    
    runtime = time.time() - start_time
    
    return q, policy, policy_grid, visited_mirror, episode_lengths, cumulative_rewards, runtime

def exploring_starts_mc_tracked(world, gamma_val=0.9, no_of_episodes=10):
    """Modified exploring_starts_mc that tracks performance metrics"""
    global gamma
    gamma = gamma_val
    
    policy_grid = np.full((world.width, world.height), "", dtype=object)
    q = np.full((world.width, world.height, len(ACTIONS)), 0.0)
    Returns = []
    
    for x in range(world.width):
        Returns.append([])
        for y in range(world.height):
            Returns[x].append([])
            for _ in range(len(ACTIONS)):
                Returns[x][y].append([])

    episode_lengths = []
    cumulative_rewards = []
    
    start_time = time.time()
    
    for _ in range(no_of_episodes):
        # Random start state
        x, y = random.randint(0, world.width - 1), random.randint(0, world.height - 1)
        while world.is_terminal(x, y) or world.is_obstacle(x, y):
            x, y = random.randint(0, world.width - 1), random.randint(0, world.height - 1)
        
        # Create deterministic policy based on current Q values
        policy = lambda x, y: {k : 1 if k == ACTIONS[np.argmax(q[x, y, :])] else 0 for k in ACTIONS}
        
        episode = generate_episode(world=world, policy=policy, start_state=(x, y), random_first_move=1)
        episode_lengths.append(len(episode))
        
        total_reward = sum([r for _, _, r in episode])
        cumulative_rewards.append(total_reward)
        
        g = 0.0
        for idx, (s, a, r) in reversed(list(enumerate(episode))):
            x, y = s
            g = r + gamma * g
            if (s, a) not in [(state, action) for state, action, _ in episode[:idx]]:
                a_idx = ACTIONS.index(a)
                Returns[x][y][a_idx].append(g)
                q[x, y, a_idx] = np.mean(Returns[x][y][a_idx])
                policy_grid[x, y] = ACTIONS[np.argmax(q[x, y, :])]
    
    runtime = time.time() - start_time
    
    return q, policy_grid, episode_lengths, cumulative_rewards, runtime

def first_visit_mc_tracked(world, policy, gamma_val=0.9, no_of_episodes=10):
    """Modified first_visit_mc that tracks performance metrics"""
    global gamma
    gamma = gamma_val
    
    v = np.full((world.width, world.height), 0.0)
    Returns = []
    for x in range(world.width):
        Returns.append([])
        for _ in range(world.height):
            Returns[x].append([])

    episode_lengths = []
    cumulative_rewards = []
    
    start_time = time.time()
    
    for _ in range(no_of_episodes):
        episode = generate_episode(world=world, policy=policy, start_state=(0, 0))
        episode_lengths.append(len(episode))
        
        total_reward = sum([r for _, _, r in episode])
        cumulative_rewards.append(total_reward)
        
        g = 0.0
        for idx, (s, _, r) in reversed(list(enumerate(episode))):
            x, y = s
            g = r + gamma * g
            if s not in [state for state, _, _ in episode[:idx]]:
                Returns[x][y].append(g)
                v[s] = np.mean(Returns[x][y])
    
    runtime = time.time() - start_time
    
    return v, episode_lengths, cumulative_rewards, runtime

# Experiment configurations
epsilon_values = [0.1, 0.3, 0.5, 0.7, 0.9]
gamma_values = [0.5, 0.7, 0.9, 0.95, 0.99]
episode_counts = [100, 500, 1000, 2000, 5000]

# Results storage
results = {
    'epsilon_comparison': {},
    'gamma_comparison': {},
    'episodes_comparison': {}
}

print("=" * 80)
print("MONTE CARLO ALGORITHM BENCHMARKING")
print("=" * 80)

# Experiment 1: Varying Epsilon (with fixed gamma=0.9, episodes=1000)
print("\n1. TESTING DIFFERENT EPSILON VALUES")
print("-" * 80)
world = create_test_world()
for eps in epsilon_values:
    print(f"\nRunning with epsilon={eps:.2f}...")
    _, _, _, _, ep_lens, cum_rewards, runtime = soft_first_visit_mc_tracked(
        world, epsilon=eps, gamma_val=0.9, no_of_episodes=1000, start_state=(5, 5)
    )
    results['epsilon_comparison'][eps] = {
        'episode_lengths': ep_lens,
        'cumulative_rewards': cum_rewards,
        'runtime': runtime,
        'avg_episode_length': np.mean(ep_lens),
        'final_avg_reward': np.mean(cum_rewards[-100:])
    }
    print(f"  Runtime: {runtime:.2f}s | Avg Episode Length: {np.mean(ep_lens):.1f} | Final Avg Reward: {np.mean(cum_rewards[-100:]):.2f}")

# Experiment 2: Varying Gamma (with fixed epsilon=0.3, episodes=1000)
print("\n2. TESTING DIFFERENT GAMMA VALUES")
print("-" * 80)
world = create_test_world()
for gam in gamma_values:
    print(f"\nRunning with gamma={gam:.2f}...")
    _, _, _, _, ep_lens, cum_rewards, runtime = soft_first_visit_mc_tracked(
        world, epsilon=0.3, gamma_val=gam, no_of_episodes=1000, start_state=(5, 5)
    )
    results['gamma_comparison'][gam] = {
        'episode_lengths': ep_lens,
        'cumulative_rewards': cum_rewards,
        'runtime': runtime,
        'avg_episode_length': np.mean(ep_lens),
        'final_avg_reward': np.mean(cum_rewards[-100:])
    }
    print(f"  Runtime: {runtime:.2f}s | Avg Episode Length: {np.mean(ep_lens):.1f} | Final Avg Reward: {np.mean(cum_rewards[-100:]):.2f}")

# Experiment 3: Varying Episode Counts (with fixed epsilon=0.3, gamma=0.9)
print("\n3. TESTING DIFFERENT EPISODE COUNTS")
print("-" * 80)
for ep_count in episode_counts:
    print(f"\nRunning with {ep_count} episodes...")
    world = create_test_world()
    _, _, _, _, ep_lens, cum_rewards, runtime = soft_first_visit_mc_tracked(
        world, epsilon=0.3, gamma_val=0.9, no_of_episodes=ep_count, start_state=(5, 5)
    )
    results['episodes_comparison'][ep_count] = {
        'episode_lengths': ep_lens,
        'cumulative_rewards': cum_rewards,
        'runtime': runtime,
        'avg_episode_length': np.mean(ep_lens),
        'final_avg_reward': np.mean(cum_rewards[-min(100, ep_count):])
    }
    print(f"  Runtime: {runtime:.2f}s | Avg Episode Length: {np.mean(ep_lens):.1f} | Final Avg Reward: {np.mean(cum_rewards[-min(100, ep_count):]):.2f}")

# Experiment 4: Compare algorithms (First-Visit MC vs Soft First-Visit MC vs Exploring Starts MC)
print("\n4. ALGORITHM COMPARISON")
print("-" * 80)
world = create_test_world()

print("\nRunning First-Visit MC (prediction)...")
v_pred, ep_lens_pred, cum_rewards_pred, runtime_pred = first_visit_mc_tracked(
    world, equiprobable_random_policy, gamma_val=0.9, no_of_episodes=1000
)
print(f"  Runtime: {runtime_pred:.2f}s | Avg Episode Length: {np.mean(ep_lens_pred):.1f}")

print("\nRunning Soft First-Visit MC (on-policy control)...")
_, _, _, _, ep_lens_ctrl, cum_rewards_ctrl, runtime_ctrl = soft_first_visit_mc_tracked(
    world, epsilon=0.3, gamma_val=0.9, no_of_episodes=1000, start_state=(5, 5)
)
print(f"  Runtime: {runtime_ctrl:.2f}s | Avg Episode Length: {np.mean(ep_lens_ctrl):.1f}")

print("\nRunning Exploring Starts MC (off-policy control)...")
_, _, ep_lens_es, cum_rewards_es, runtime_es = exploring_starts_mc_tracked(
    world, gamma_val=0.9, no_of_episodes=1000
)
print(f"  Runtime: {runtime_es:.2f}s | Avg Episode Length: {np.mean(ep_lens_es):.1f}")

# Replace the existing plotting section with this improved version:

# PLOTTING RESULTS
print("\n" + "=" * 80)
print("GENERATING PLOTS")
print("=" * 80)

fig = plt.figure(figsize=(16, 8))

# Plot 1: Epsilon Comparison - Episode Lengths
ax1 = plt.subplot(2, 2, 1)
for eps, data in results['epsilon_comparison'].items():
    smoothed = np.convolve(data['episode_lengths'], np.ones(50)/50, mode='valid')
    ax1.plot(smoothed, label=f'ε={eps:.2f}', linewidth=2)
ax1.set_xlabel('Episode', fontsize=11)
ax1.set_ylabel('Episode Length (smoothed)', fontsize=11)
ax1.set_title('Effect of Epsilon on Episode Length', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Epsilon Comparison - Runtime (LINE PLOT)
ax2 = plt.subplot(2, 2, 2)
epsilons = sorted(list(results['epsilon_comparison'].keys()))
runtimes = [results['epsilon_comparison'][e]['runtime'] for e in epsilons]
ax2.plot(epsilons, runtimes, marker='o', linewidth=2.5, markersize=8, 
         color='steelblue', markerfacecolor='lightblue', markeredgewidth=2)
ax2.set_xlabel('Epsilon (ε)', fontsize=11)
ax2.set_ylabel('Runtime (seconds)', fontsize=11)
ax2.set_title('Runtime vs Epsilon', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(epsilons)

# Plot 3: Algorithm Comparison - All Three Algorithms
ax3 = plt.subplot(2, 2, 3)
window_size = 50
if len(ep_lens_pred) >= window_size:
    smoothed_pred = np.convolve(ep_lens_pred, np.ones(window_size)/window_size, mode='valid')
    ax3.plot(range(len(smoothed_pred)), smoothed_pred, label='First-Visit MC', 
             linewidth=2.5, color='blue', alpha=0.8)
if len(ep_lens_ctrl) >= window_size:
    smoothed_ctrl = np.convolve(ep_lens_ctrl, np.ones(window_size)/window_size, mode='valid')
    ax3.plot(range(len(smoothed_ctrl)), smoothed_ctrl, label='Soft First-Visit MC', 
             linewidth=2.5, color='red', alpha=0.8)
if len(ep_lens_es) >= window_size:
    smoothed_es = np.convolve(ep_lens_es, np.ones(window_size)/window_size, mode='valid')
    ax3.plot(range(len(smoothed_es)), smoothed_es, label='Exploring Starts MC', 
             linewidth=2.5, color='green', alpha=0.8)
ax3.set_xlabel('Episode', fontsize=11)
ax3.set_ylabel('Episode Length (smoothed)', fontsize=11)
ax3.set_title('Algorithm Comparison: All Three Algorithms', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Algorithm Comparison - First-Visit MC vs Soft First-Visit MC Only
ax4 = plt.subplot(2, 2, 4)
if len(ep_lens_pred) >= window_size:
    smoothed_pred = np.convolve(ep_lens_pred, np.ones(window_size)/window_size, mode='valid')
    ax4.plot(range(len(smoothed_pred)), smoothed_pred, label='First-Visit MC', 
             linewidth=2.5, color='blue', alpha=0.8)
if len(ep_lens_ctrl) >= window_size:
    smoothed_ctrl = np.convolve(ep_lens_ctrl, np.ones(window_size)/window_size, mode='valid')
    ax4.plot(range(len(smoothed_ctrl)), smoothed_ctrl, label='Soft First-Visit MC', 
             linewidth=2.5, color='red', alpha=0.8)
ax4.set_xlabel('Episode', fontsize=11)
ax4.set_ylabel('Episode Length (smoothed)', fontsize=11)
ax4.set_title('Algorithm Comparison: First-Visit vs Soft First-Visit', fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('monte_carlo_benchmarks.png', dpi=300, bbox_inches='tight')
print("\nPlot saved as 'monte_carlo_benchmarks.png'")
plt.show()

# Summary Statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print("\nBest Epsilon (lowest avg episode length):")
best_eps = min(results['epsilon_comparison'].items(), key=lambda x: x[1]['avg_episode_length'])
print(f"  ε = {best_eps[0]:.2f} with avg episode length: {best_eps[1]['avg_episode_length']:.2f}")

print("\nAlgorithm Performance Comparison:")
print(f"  First-Visit MC (Prediction):")
print(f"    Runtime: {runtime_pred:.2f}s")
print(f"    Avg Episode Length: {np.mean(ep_lens_pred):.1f}")
print(f"    Final Avg Episode Length (last 100): {np.mean(ep_lens_pred[-100:]):.1f}")
print(f"\n  Soft First-Visit MC (On-Policy Control):")
print(f"    Runtime: {runtime_ctrl:.2f}s")
print(f"    Avg Episode Length: {np.mean(ep_lens_ctrl):.1f}")
print(f"    Final Avg Episode Length (last 100): {np.mean(ep_lens_ctrl[-100:]):.1f}")
print(f"\n  Exploring Starts MC (Off-Policy Control):")
print(f"    Runtime: {runtime_es:.2f}s")
print(f"    Avg Episode Length: {np.mean(ep_lens_es):.1f}")
print(f"    Final Avg Episode Length (last 100): {np.mean(ep_lens_es[-100:]):.1f}")

print("\n" + "=" * 80)

print("\n" + "=" * 80)
print("DETAILED EPISODE LENGTH ANALYSIS")
print("=" * 80)

print("\nFirst-Visit MC (Prediction):")
print(f"  Episodes hitting max_steps (1000): {sum(1 for x in ep_lens_pred if x >= 1000)}/{len(ep_lens_pred)}")
print(f"  Min: {min(ep_lens_pred)}, Max: {max(ep_lens_pred)}, Avg: {np.mean(ep_lens_pred):.1f}")
print(f"  Episodes < 1000: {[x for x in ep_lens_pred if x < 1000][:10]}...")  # Show first 10

print("\nSoft First-Visit MC (On-Policy Control):")
print(f"  Episodes hitting max_steps (1000): {sum(1 for x in ep_lens_ctrl if x >= 1000)}/{len(ep_lens_ctrl)}")
print(f"  Min: {min(ep_lens_ctrl)}, Max: {max(ep_lens_ctrl)}, Avg: {np.mean(ep_lens_ctrl):.1f}")

print("\nExploring Starts MC (Off-Policy Control):")
print(f"  Episodes hitting max_steps (1000): {sum(1 for x in ep_lens_es if x >= 1000)}/{len(ep_lens_es)}")
print(f"  Min: {min(ep_lens_es)}, Max: {max(ep_lens_es)}, Avg: {np.mean(ep_lens_es):.1f}")

MONTE CARLO ALGORITHM BENCHMARKING

1. TESTING DIFFERENT EPSILON VALUES
--------------------------------------------------------------------------------

Running with epsilon=0.10...
  Runtime: 0.80s | Avg Episode Length: 16.4 | Final Avg Reward: 10.00

Running with epsilon=0.30...
  Runtime: 0.31s | Avg Episode Length: 14.6 | Final Avg Reward: 10.00

Running with epsilon=0.50...
  Runtime: 0.37s | Avg Episode Length: 19.7 | Final Avg Reward: 10.00

Running with epsilon=0.70...
  Runtime: 0.45s | Avg Episode Length: 27.8 | Final Avg Reward: 10.00

Running with epsilon=0.90...
  Runtime: 1.09s | Avg Episode Length: 62.8 | Final Avg Reward: 10.00

2. TESTING DIFFERENT GAMMA VALUES
--------------------------------------------------------------------------------

Running with gamma=0.50...
  Runtime: 0.28s | Avg Episode Length: 13.5 | Final Avg Reward: 10.00

Running with gamma=0.70...
  Runtime: 0.29s | Avg Episode Length: 13.9 | Final Avg Reward: 10.00

Running with gamma=0.90...
  Runti

KeyboardInterrupt: 

Try to experiment with your implementation by running it on different world sizes (be careful not to make your world too large or you should have multiple terminals that an agent is likely to hit, otherwise it may take too long), try to experiment with different numbers of episodes, and different values of epsilon:

In [ ]:
### TODO: Implement your code here

### Optional exercise

Try to implement exploring starts (see page 99 of [Introduction to Reinforcement Learning](http://incompleteideas.net/book/the-book-2nd.html) for the algorithm). It should be straightforward and only require minimal changes to the code for the exercise above.